# f-I Curve Analysis — Burton & Urban (2014) Comparison

Runs somatic current-clamp experiments on the Birgiolas 2020 MC1–MC5 and TC1–TC5
isolated cell models and reproduces the electrophysiological characterization from:

> Burton, S.D. & Urban, N.N. (2014). Intrinsic cellular differences drive parallel
> processing in olfaction. *Journal of Physiology*, 592(10), 2097–2118.

**Figures reproduced (Figure 5)**
- **5A/B** — Stacked voltage traces at multiple current amplitudes (representative MC1, TC1)
- **5C/D** — Individual f-I curves for all models overlaid per cell type
- **5E**   — Mean ± SEM f-I comparison between MC and TC
- **5F**   — f-I gain vs input resistance scatter

**Tables reproduced**
- **Table 4** — Action potential properties (threshold, amplitude, FWHM, slopes, AHP)
- **Table 5** — Spike train properties (rheobase, latency, peak rate, gain, CV_ISI)

**Protocol** (Burton & Urban 2014, Methods):
- Depolarising steps: 2 s, 0–300 pA in 50 pA increments, cell re-initialised each run
- Hyperpolarising steps: 2 s, 0 to −300 pA in 50 pA decrements (for input resistance)
- Synaptic blockers not needed — models are isolated cells with no network input


## Setup

In [ ]:
import sys
import os

repo_root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root_path not in sys.path:
    sys.path.insert(0, repo_root_path)

import numpy as np
import matplotlib.pyplot as plt

from single_cell_utils import (
    run_current_clamp,
    run_current_clamp_series,
    run_hyperpolarizing_steps,
    find_bias_current,
)
from fi_curve_utils import (
    traces_to_fi,
    compute_fi_slope,
    find_spike_times_milliseconds,
    compute_interspike_interval_statistics,
    compute_rheobase_nanoamps,
    compute_rheobase_spike_latency_milliseconds,
    compute_action_potential_properties,
    compute_input_resistance_megaohms,
    plot_voltage_traces_from_results,
    plot_fi_overlay_by_type,
    plot_fi_mean_sem_comparison,
    plot_gain_vs_input_resistance_scatter,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 9})


### Protocol Parameters

All tunable constants in one place. Edit here to change the current range,
step duration, or detection threshold for the entire notebook.

In [ ]:
CELL_TYPES = ["MC", "TC"]
CELL_TYPE_COUNT = 5
CELL_MODEL_NAMES = [f"{cell_type}{enumeration}" for cell_type in CELL_TYPES for enumeration in range(1, CELL_TYPE_COUNT+1)]
print(CELL_MODEL_NAMES)

# ── Depolarising step protocol (Burton & Urban 2014, Figure 5) ───────────────
STEP_CURRENT_START_NANOAMPS     = 0.0    # 0 pA
STEP_CURRENT_STOP_NANOAMPS      = 0.30   # 300 pA
STEP_CURRENT_INCREMENT_NANOAMPS = 0.05   # 50 pA increments

STEP_DURATION_MILLISECONDS = 2000.0  # 2-second steps, matching paper
STEP_DELAY_MILLISECONDS    = 200.0   # silent lead-in before step onset
TAIL_DURATION_MILLISECONDS = 200.0   # silent tail after step offset

SIMULATION_TIMESTEP_MILLISECONDS = 0.1
SIMULATION_TEMPERATURE_CELSIUS   = 35.0

# ── Spike and action-potential detection ─────────────────────────────────────
SPIKE_DETECTION_THRESHOLD_MILLIVOLTS               = -20.0
AP_DERIVATIVE_THRESHOLD_MILLIVOLTS_PER_MILLISECOND = 20.0  # paper criterion: 20 mV/ms

# ── Hyperpolarising steps (input resistance, Burton & Urban 2014) ─────────────
HYPERPOLARIZING_CURRENT_START_NANOAMPS     = 0.0
HYPERPOLARIZING_CURRENT_STOP_NANOAMPS      = -0.30   # -300 pA
HYPERPOLARIZING_CURRENT_INCREMENT_NANOAMPS = -0.05   # -50 pA steps
HYPERPOLARIZING_STEP_DURATION_MILLISECONDS = 2000.0

# ── Paper reference value ────────────────────────────────────────────────────
PAPER_TARGET_MEMBRANE_POTENTIAL_MILLIVOLTS = -58.0  # paper normalises V_m here


---
## 1. Resting Membrane Potential Survey

Measure each model's resting potential with zero injected current (500 ms simulation).
The paper normalises V_m to −58 mV via a DC bias current before each step; large offsets
from −58 mV may affect quantitative comparisons. Call `find_bias_current(cell_name)` to
compute and apply the correction if needed.


In [ ]:
resting_potential_by_cell_name = {}

print(f"{'Cell':<6}  {'Resting V_m (mV)':>18}  {'Offset from -58 mV':>20}")
print("-" * 48)

for cell_name in CELL_MODEL_NAMES:
    zero_current_trace = run_current_clamp(
        cell_name,
        amp_nA=0.0,
        duration_ms=500.0,
        delay_ms=0.0,
        tail_ms=0.0,
        dt=SIMULATION_TIMESTEP_MILLISECONDS,
        celsius=SIMULATION_TEMPERATURE_CELSIUS,
        use_coreneuron=True,
        use_gpu=True,
    )
    resting_potential_millivolts = float(zero_current_trace["v_soma"][-1])
    resting_potential_by_cell_name[cell_name] = resting_potential_millivolts
    offset_from_paper_target_millivolts = (
        resting_potential_millivolts - PAPER_TARGET_MEMBRANE_POTENTIAL_MILLIVOLTS
    )
    print(
        f"{cell_name:<6}  "
        f"{resting_potential_millivolts:>18.1f}  "
        f"{offset_from_paper_target_millivolts:>+20.1f}"
    )

---
## 2. Depolarising Step Protocol

One independent current-clamp simulation per amplitude for every cell model.
Protocol matches Burton & Urban (2014): 2 s steps, 0–300 pA in 50 pA increments.
The cell state is re-initialised at the start of each amplitude run.


In [ ]:
current_amplitude_steps_nanoamps = np.arange(
    STEP_CURRENT_START_NANOAMPS,
    STEP_CURRENT_STOP_NANOAMPS + STEP_CURRENT_INCREMENT_NANOAMPS * 0.5,
    STEP_CURRENT_INCREMENT_NANOAMPS,
)

step_traces_by_cell_name = {}
fi_curve_data_by_cell_name = {}

for cell_name in CELL_MODEL_NAMES:
    print(f"{cell_name}...", end=" ", flush=True)

    cell_step_traces = run_current_clamp_series(
        cell_name,
        amps_nA=current_amplitude_steps_nanoamps,
        duration_ms=STEP_DURATION_MILLISECONDS,
        delay_ms=STEP_DELAY_MILLISECONDS,
        tail_ms=TAIL_DURATION_MILLISECONDS,
        dt=SIMULATION_TIMESTEP_MILLISECONDS,
        celsius=SIMULATION_TEMPERATURE_CELSIUS,
        use_coreneuron=True,
        use_gpu=True,
    )
    step_traces_by_cell_name[cell_name] = cell_step_traces

    current_amplitudes_nanoamps, firing_rates_hertz = traces_to_fi(
        cell_step_traces,
        STEP_DURATION_MILLISECONDS,
        threshold_mV=SPIKE_DETECTION_THRESHOLD_MILLIVOLTS,
    )
    fi_curve_data_by_cell_name[cell_name] = dict(
        current_amplitudes_nanoamps=current_amplitudes_nanoamps,
        firing_rates_hertz=firing_rates_hertz,
    )

    rheobase_current_nanoamps = compute_rheobase_nanoamps(
        cell_step_traces, SPIKE_DETECTION_THRESHOLD_MILLIVOLTS
    )
    if not np.isnan(rheobase_current_nanoamps):
        print(f"rheobase ~ {rheobase_current_nanoamps * 1000.0:.0f} pA")
    else:
        print("no spikes detected")

print("\nStep protocol complete.")

---
## 3. Hyperpolarising Steps (Input Resistance)

Estimates somatic input resistance from ΔV / ΔI using hyperpolarising current steps.
Protocol matches Burton & Urban (2014): 0 to −300 pA in 50 pA decrements, 2 s each.


In [ ]:
hyperpolarizing_traces_by_cell_name = {}

for cell_name in CELL_MODEL_NAMES:
    print(f"{cell_name}...", end=" ", flush=True)

    cell_hyperpolarizing_traces = run_hyperpolarizing_steps(
        cell_name,
        current_start_nanoamps=HYPERPOLARIZING_CURRENT_START_NANOAMPS,
        current_stop_nanoamps=HYPERPOLARIZING_CURRENT_STOP_NANOAMPS,
        current_step_nanoamps=HYPERPOLARIZING_CURRENT_INCREMENT_NANOAMPS,
        step_duration_milliseconds=HYPERPOLARIZING_STEP_DURATION_MILLISECONDS,
        delay_milliseconds=STEP_DELAY_MILLISECONDS,
        tail_duration_milliseconds=TAIL_DURATION_MILLISECONDS,
        timestep_milliseconds=SIMULATION_TIMESTEP_MILLISECONDS,
        temperature_celsius=SIMULATION_TEMPERATURE_CELSIUS,
        use_coreneuron=True,
        use_gpu=True,
    )
    hyperpolarizing_traces_by_cell_name[cell_name] = cell_hyperpolarizing_traces

    estimated_input_resistance_megaohms = compute_input_resistance_megaohms(
        cell_hyperpolarizing_traces,
        step_duration_milliseconds=HYPERPOLARIZING_STEP_DURATION_MILLISECONDS,
        delay_milliseconds=STEP_DELAY_MILLISECONDS,
    )
    print(f"R_input ~ {estimated_input_resistance_megaohms:.1f} MΩ")

print("\nHyperpolarising protocol complete.")

---
## 4. Compute All Metrics

Extract paper-comparable metrics from the traces collected above. Results are
stored in `all_cell_metrics` (list of per-cell dicts) and used by all plot and
table cells below.

**Note on CV_ISI**: The paper reports CV_ISI measured at −20 Hz from the peak
firing rate. Here we compute CV_ISI from the step with the most spikes as an
approximation — this is acceptable for the small number of models per type.


In [ ]:
all_cell_metrics = []

for cell_name in CELL_MODEL_NAMES:
    cell_step_traces = step_traces_by_cell_name[cell_name]
    cell_hyperpolarizing_traces = hyperpolarizing_traces_by_cell_name[cell_name]
    fi_data = fi_curve_data_by_cell_name[cell_name]

    # f-I gain (convert from Hz/nA to Hz/50 pA = Hz/nA divided by 20)
    fi_slope_hertz_per_nanoamp, _, _ = compute_fi_slope(
        fi_data["current_amplitudes_nanoamps"],
        fi_data["firing_rates_hertz"],
    )
    fi_gain_hertz_per_50_picoamps = (
        fi_slope_hertz_per_nanoamp / 20.0
        if not np.isnan(fi_slope_hertz_per_nanoamp)
        else np.nan
    )

    # Rheobase
    rheobase_current_nanoamps = compute_rheobase_nanoamps(
        cell_step_traces, SPIKE_DETECTION_THRESHOLD_MILLIVOLTS
    )

    # Spike latency at rheobase
    rheobase_spike_latency_milliseconds = compute_rheobase_spike_latency_milliseconds(
        cell_step_traces,
        SPIKE_DETECTION_THRESHOLD_MILLIVOLTS,
        STEP_DELAY_MILLISECONDS,
    )

    # ISI statistics — use the step with the most spikes
    traces_with_multiple_spikes = [
        trace for trace in cell_step_traces
        if len(find_spike_times_milliseconds(
            trace, SPIKE_DETECTION_THRESHOLD_MILLIVOLTS, STEP_DELAY_MILLISECONDS
        )) >= 2
    ]
    if traces_with_multiple_spikes:
        most_active_trace = max(
            traces_with_multiple_spikes,
            key=lambda trace: len(find_spike_times_milliseconds(
                trace, SPIKE_DETECTION_THRESHOLD_MILLIVOLTS, STEP_DELAY_MILLISECONDS
            )),
        )
        isi_statistics = compute_interspike_interval_statistics(
            most_active_trace,
            SPIKE_DETECTION_THRESHOLD_MILLIVOLTS,
            STEP_DELAY_MILLISECONDS,
        )
    else:
        isi_statistics = dict(
            peak_instantaneous_firing_rate_hertz=np.nan,
            coefficient_of_variation_interspike_interval=np.nan,
        )

    # AP shape properties at rheobase (first AP at weakest suprathreshold input)
    if not np.isnan(rheobase_current_nanoamps):
        rheobase_trace = next(
            (
                trace for trace in cell_step_traces
                if np.isclose(float(trace["amp_nA"]), rheobase_current_nanoamps)
            ),
            None,
        )
        action_potential_properties = (
            compute_action_potential_properties(
                rheobase_trace,
                AP_DERIVATIVE_THRESHOLD_MILLIVOLTS_PER_MILLISECOND,
                STEP_DELAY_MILLISECONDS,
            )
            if rheobase_trace is not None
            else {}
        )
    else:
        action_potential_properties = {}

    # Input resistance
    input_resistance_megaohms = compute_input_resistance_megaohms(
        cell_hyperpolarizing_traces,
        step_duration_milliseconds=HYPERPOLARIZING_STEP_DURATION_MILLISECONDS,
        delay_milliseconds=STEP_DELAY_MILLISECONDS,
    )

    all_cell_metrics.append(dict(
        cell_name=cell_name,
        cell_type=cell_type,
        fi_gain_hertz_per_50_picoamps=fi_gain_hertz_per_50_picoamps,
        rheobase_picoamps=(
            rheobase_current_nanoamps * 1000.0
            if not np.isnan(rheobase_current_nanoamps)
            else np.nan
        ),
        rheobase_spike_latency_milliseconds=rheobase_spike_latency_milliseconds,
        peak_instantaneous_firing_rate_hertz=(
            isi_statistics["peak_instantaneous_firing_rate_hertz"]
        ),
        coefficient_of_variation_interspike_interval=(
            isi_statistics["coefficient_of_variation_interspike_interval"]
        ),
        input_resistance_megaohms=input_resistance_megaohms,
        is_representative_cell=(cell_number == 1),
        action_potential_properties=action_potential_properties,
    ))

print(f"Metrics computed for {len(all_cell_metrics)} cell models.")


---
## Figure 5A/B — Stacked Voltage Traces

Equivalent to Figure 5A/B in Burton & Urban (2014): stacked soma voltage traces
at each current amplitude for a representative MC (MC1) and TC (TC1).


In [ ]:
# Select a subset of current amplitudes matching the paper's figure (50–300 pA)
displayed_current_amplitudes_nanoamps = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

for cell_name in CELL_MODEL_NAMES:
    # Filter the full trace list to the displayed amplitudes
    full_traces = step_traces_by_cell_name[cell_name]
    displayed_traces = [
        trace for trace in full_traces
        if any(
            np.isclose(float(trace["amp_nA"]), target_amplitude)
            for target_amplitude in displayed_current_amplitudes_nanoamps
        )
    ]
    displayed_traces.sort(key=lambda trace: trace["amp_nA"])

    figure = plot_voltage_traces_from_results(
        displayed_traces,
        cell_name=cell_name,
        step_duration_milliseconds=STEP_DURATION_MILLISECONDS,
        delay_milliseconds=STEP_DELAY_MILLISECONDS,
    )
    plt.show()


---
## Figure 5C/D — f-I Curves Overlaid by Cell Type

Equivalent to Figure 5C/D: all model f-I curves for MC (left) and TC (right),
plotted as individual thin lines with the first model (MC1, TC1) drawn thicker
as the representative trace.


In [ ]:
CELL_TYPES = ["MC", "TC"]
results_by_type = {
    cell_type: {
        f"{cell_type}{cell_number}": fi_curve_data_by_cell_name[f"{cell_type}{cell_number}"]
        for cell_number in range(1, CELL_TYPE_COUNT + 1)
    }
    for cell_type in CELL_TYPES
}

figure = plot_fi_overlay_by_type(results_by_type)
plt.show()


---
## Figure 5E — Average f-I Comparison (Mean ± SEM)

Equivalent to Figure 5E: mean firing rate ± SEM for MC (n=5) and TC (n=5)
plotted on the same axes. The shaded band represents ±1 SEM.


In [ ]:
figure = plot_fi_mean_sem_comparison(results_by_type)
plt.show()


---
## Figure 5F — f-I Gain vs Input Resistance

Equivalent to Figure 5F: scatter of f-I gain (Hz/50 pA) vs somatic input
resistance (MΩ) for all MC and TC models. MC = '+', TC = inverted triangle.
The first model of each type is drawn larger (representative).


In [ ]:
figure = plot_gain_vs_input_resistance_scatter(all_cell_metrics)
plt.show()


---
## Table 4 — Action Potential Properties

Comparison with Burton & Urban (2014) Table 4: V_threshold, amplitude, FWHM,
rising slope, falling slope, AHP amplitude, and T_AHP50% for MC and TC models.

Paper values (mean ± SD):
| Property | Mitral cells | Tufted cells | P |
|---|---|---|---|
| V_threshold (mV) | −42.2 ± 3.0 | −42.5 ± 2.9 | n.s. |
| Amplitude (mV) | 76.2 ± 5.4 | 72.1 ± 5.5 | n.s. |
| FWHM (ms) | 1.06 ± 0.20 | 0.87 ± 0.10 | 0.03 |
| Rise slope (mV/ms) | 237.9 ± 48.4 | 197.9 ± 62.5 | n.s. |
| Fall slope (mV/ms) | −72.2 ± 20.4 | −91.4 ± 13.0 | 0.03 |
| AHP (mV) | 14.8 ± 3.2 | 16.8 ± 3.3 | n.s. |
| T_AHP50% (ms) | 58.2 ± 77.5 | 20.5 ± 20.1 | 5.8×10⁻³ |


In [ ]:
import pandas as pd

table4_rows = []
for cell_metrics in all_cell_metrics:
    ap_props = cell_metrics["action_potential_properties"]
    table4_rows.append(dict(
        Cell=cell_metrics["cell_name"],
        Type=cell_metrics["cell_type"],
        AP_onset_mV=round(ap_props.get("ap_onset_millivolts", float("nan")), 1),
        Amplitude_mV=round(ap_props.get("ap_amplitude_millivolts", float("nan")), 1),
        FWHM_ms=round(ap_props.get("ap_full_width_half_maximum_milliseconds", float("nan")), 2),
        Rise_slope_mV_per_ms=round(
            ap_props.get("ap_rise_slope_millivolts_per_millisecond", float("nan")), 1
        ),
        Fall_slope_mV_per_ms=round(
            ap_props.get("ap_fall_slope_millivolts_per_millisecond", float("nan")), 1
        ),
        AHP_amplitude_mV=round(ap_props.get("ahp_amplitude_millivolts", float("nan")), 1),
        T_AHP50_ms=round(ap_props.get("ahp_half_decay_time_milliseconds", float("nan")), 1),
    ))

table4_dataframe = pd.DataFrame(table4_rows).set_index("Cell")

# Summary rows (mean per type)
for cell_type_name in CELL_TYPES:
    type_subset = table4_dataframe[table4_dataframe["Type"] == cell_type_name].drop(
        columns="Type"
    )
    type_mean_row = type_subset.mean().rename(f"{cell_type_name} mean")
    type_std_row = type_subset.std().rename(f"{cell_type_name} SD")
    print(f"\n{cell_type_name} mean:")
    print(type_mean_row.to_string())
    print(f"\n{cell_type_name} SD:")
    print(type_std_row.to_string())

print("\nFull table:")
display(table4_dataframe)


---
## Table 5 — Spike Train Properties

Comparison with Burton & Urban (2014) Table 5: rheobase, rheobase spike latency,
peak instantaneous firing rate, f-I curve gain, and CV_ISI.

Paper values (mean ± SD):
| Property | Mitral cells | Tufted cells | P |
|---|---|---|---|
| Rheobase (pA) | 111.4 ± 55.7 | 94.6 ± 49.7 | n.s. |
| Spike latency (ms) | 510.0 ± 486.0 | 402.3 ± 479.5 | n.s. |
| Peak rate (Hz) | 62.8 ± 15.9 | 120.1 ± 28.4 | 9.5×10⁻¹¹ |
| FI gain (Hz/50 pA) | 9.8 ± 3.8 | 20.3 ± 7.2 | 1.5×10⁻⁸ |
| CV_ISI | 0.45 ± 0.29 | 0.80 ± 0.43 | 3.3×10⁻⁴ |


In [ ]:
table5_rows = []
for cell_metrics in all_cell_metrics:
    table5_rows.append(dict(
        Cell=cell_metrics["cell_name"],
        Type=cell_metrics["cell_type"],
        Rheobase_pA=round(cell_metrics["rheobase_picoamps"], 1),
        Spike_latency_ms=round(cell_metrics["rheobase_spike_latency_milliseconds"], 1),
        Peak_rate_Hz=round(cell_metrics["peak_instantaneous_firing_rate_hertz"], 1),
        FI_gain_Hz_per_50pA=round(cell_metrics["fi_gain_hertz_per_50_picoamps"], 1),
        CV_ISI=round(cell_metrics["coefficient_of_variation_interspike_interval"], 3),
    ))

table5_dataframe = pd.DataFrame(table5_rows).set_index("Cell")

for cell_type_name in CELL_TYPES:
    type_subset = table5_dataframe[table5_dataframe["Type"] == cell_type_name].drop(
        columns="Type"
    )
    type_mean_row = type_subset.mean().rename(f"{cell_type_name} mean")
    type_std_row = type_subset.std().rename(f"{cell_type_name} SD")
    print(f"\n{cell_type_name} mean:")
    print(type_mean_row.to_string())
    print(f"\n{cell_type_name} SD:")
    print(type_std_row.to_string())

print("\nFull table:")
display(table5_dataframe)


---
## 3.6-Second f-I Curves — Overlay by Cell Type

Same 50 pA increment protocol as above but with **3.6-second current steps** and the range extended well past 300 pA to capture saturation.
Longer steps reveal steady-state firing rates and spike-frequency adaptation that the 2 s protocol misses.

In [ ]:
STEP_DURATION_10S_MS           = 3_600.0    # 3.6-second steps (vs 2 s above)
STEP_CURRENT_STOP_10S_NANOAMPS = 1.0        # 1000 pA — extend past 300 pA to capture saturation

current_amps_10s = np.arange(
    STEP_CURRENT_START_NANOAMPS,
    STEP_CURRENT_STOP_10S_NANOAMPS + STEP_CURRENT_INCREMENT_NANOAMPS * 0.5,
    STEP_CURRENT_INCREMENT_NANOAMPS,
)

fi_data_10s_by_cell = {}

for cell_name in CELL_MODEL_NAMES:
    print(f"{cell_name}...", end=" ", flush=True)

    traces = run_current_clamp_series(
        cell_name,
        amps_nA=current_amps_10s,
        duration_ms=STEP_DURATION_10S_MS,
        delay_ms=STEP_DELAY_MILLISECONDS,
        tail_ms=TAIL_DURATION_MILLISECONDS,
        dt=SIMULATION_TIMESTEP_MILLISECONDS,
        celsius=SIMULATION_TEMPERATURE_CELSIUS,
        use_coreneuron=True,
        use_gpu=True,
    )

    amps, freqs = traces_to_fi(
        traces,
        STEP_DURATION_10S_MS,
        threshold_mV=SPIKE_DETECTION_THRESHOLD_MILLIVOLTS,
    )
    fi_data_10s_by_cell[cell_name] = dict(
        current_amplitudes_nanoamps=amps,
        firing_rates_hertz=freqs,
    )

    rheobase = compute_rheobase_nanoamps(traces, SPIKE_DETECTION_THRESHOLD_MILLIVOLTS)
    if not np.isnan(rheobase):
        print(f"rheobase ~ {rheobase * 1000:.0f} pA")
    else:
        print("no spikes")

print("\nStep protocol (3.6 s, 0–1000 pA) complete.")

MC1... rheobase ~ 150 pA
MC2... best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1
best_balance=nan ncell=1 ntype=1 nwarp=1


### Batched 3.6-Second f-I Sweep

If you only need firing rates, the batched current-step helper runs each
cell/current condition in a single parallelizable workload. This avoids
per-cell launch overhead and is usually much faster for single-cell f-I
sweeps.


In [ ]:
from fi_curve_utils import run_cell_type_fi, plot_fi_overlay_by_type

fi_data_10s_batch_by_type = {}

for cell_type in CELL_TYPES:
    fi_data_10s_batch_by_type[cell_type] = run_cell_type_fi(
        cell_type,
        protocol="batch",
        n_cells=CELL_TYPE_COUNT,
        i_start_nA=STEP_CURRENT_START_NANOAMPS,
        i_stop_nA=STEP_CURRENT_STOP_10S_NANOAMPS,
        i_step_nA=STEP_CURRENT_INCREMENT_NANOAMPS,
        step_dur_ms=STEP_DURATION_10S_MS,
        delay_ms=STEP_DELAY_MILLISECONDS,
        tail_ms=TAIL_DURATION_MILLISECONDS,
        dt=SIMULATION_TIMESTEP_MILLISECONDS,
        celsius=SIMULATION_TEMPERATURE_CELSIUS,
        threshold_mV=SPIKE_DETECTION_THRESHOLD_MILLIVOLTS,
        use_coreneuron=True,
        use_gpu=True,
    )

fi_data_10s_batch_by_type_for_plot = {
    cell_type: {
        cell_name: {
            "current_amplitudes_nanoamps": cell_data["currents_nA"],
            "firing_rates_hertz": cell_data["freqs_hz"],
        }
        for cell_name, cell_data in type_results.items()
    }
    for cell_type, type_results in fi_data_10s_batch_by_type.items()
}

plot_fi_overlay_by_type(fi_data_10s_batch_by_type_for_plot)

# NOTE: CoreNEURON with IClamp may require MPI launch in some
# environments, for example:
#   mpiexec -n 1 nrniv -mpi -python <notebook-runner>
# Set use_coreneuron=False if running directly in a regular Jupyter
# kernel.


In [ ]:
results_10s_by_type = {
    cell_type: {
        f"{cell_type}{i}": fi_data_10s_by_cell[f"{cell_type}{i}"]
        for i in range(1, CELL_TYPE_COUNT + 1)
    }
    for cell_type in CELL_TYPES
}

fig = plot_fi_overlay_by_type(results_10s_by_type)
plt.show()

---
## 3.6-Second f-I Curves — Continuous Staircase (50 pA steps, 0–1000 pA)

Same current range and step size as the discrete protocol above, but run as a **single continuous simulation per cell**: the current steps up by 50 pA every 3.6 s without re-initialising the cell between steps.

This mirrors how a real patch-clamp experiment is conducted. Accumulated channel state (slow inactivation, Ca²⁺ history, etc.) carries over between steps, so the f-I curve here reflects how the cell behaves in a live recording rather than in isolation from its own history.

In [ ]:
from single_cell_utils import run_fi_ramp
from fi_curve_utils import ramp_to_fi

fi_data_staircase_by_cell = {}

for cell_name in CELL_MODEL_NAMES:
    print(f"{cell_name}...", end=" ", flush=True)

    ramp = run_fi_ramp(
        cell_name,
        i_start_nA=STEP_CURRENT_START_NANOAMPS,
        i_stop_nA=STEP_CURRENT_STOP_10S_NANOAMPS,
        i_step_nA=STEP_CURRENT_INCREMENT_NANOAMPS,
        step_dur_ms=STEP_DURATION_10S_MS,
        delay_ms=STEP_DELAY_MILLISECONDS,
        tail_ms=TAIL_DURATION_MILLISECONDS,
        dt=SIMULATION_TIMESTEP_MILLISECONDS,
        celsius=SIMULATION_TEMPERATURE_CELSIUS,
        use_coreneuron=True,
        use_gpu=True,
    )

    amps, freqs = ramp_to_fi(ramp, threshold_mV=SPIKE_DETECTION_THRESHOLD_MILLIVOLTS)
    fi_data_staircase_by_cell[cell_name] = dict(
        current_amplitudes_nanoamps=amps,
        firing_rates_hertz=freqs,
    )
    print("done")

print("\nStaircase protocol complete.")

In [ ]:
results_staircase_by_type = {
    cell_type: {
        f"{cell_type}{i}": fi_data_staircase_by_cell[f"{cell_type}{i}"]
        for i in range(1, CELL_TYPE_COUNT + 1)
    }
    for cell_type in CELL_TYPES
}

fig = plot_fi_overlay_by_type(results_staircase_by_type)
plt.show()